# Probabilistic U-Net with a post-hoc consensus selection head

Reimplementation of Kohl et al., *A Probabilistic U-Net for Segmentation of
Ambiguous Images* (NeurIPS 2018, [arXiv:1806.05034](https://arxiv.org/abs/1806.05034)),
extended with a head that selects one sample without ground truth.

**This notebook demonstrates and narrates. It does not train.** The 6-page PDF carries
the argumentation; this shows the results and the qualitative behaviour.

It runs **on CPU** and needs neither a GPU nor the full 450 MB dataset:

| needs | where it comes from |
|---|---|
| `results/comparison.json` | tracked in the repo |
| `data/processed/lidc_subset.npz` | tracked in the repo (panel patches only) |
| `data/splits/split.json` | tracked in the repo |
| three weights-only `.pt` files | downloaded (cell below) |

Training is deliberately absent: three runs exceed a Colab session, and a different
device would produce numbers that are not comparable with the reported ones (seeds do
not reproduce across backends).

## 1. Setup

On Colab, clone the repository and install it. `matplotlib` is preinstalled there; the
`[notebook]` extra covers a local run.

In [ ]:
# Colab only -- skip when running from a local checkout.
# !git clone https://github.com/<user>/prob-unet-consensus-selection.git
# %cd prob-unet-consensus-selection
# !pip install -q -e ".[notebook]"


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from probunet import paths
from probunet.data.lidc import LidcArrays, panel_batch
from probunet.training.diagnostics import make_panel
from probunet.utils.runtime import describe_device, select_device

device = select_device('auto')
print('device:', describe_device(device))


## 2. Results

`comparison.json` is produced by `scripts/compare.py` and tracked in the repo. Every
number below was computed on one device in one run of that script -- no training and no
recomputation happens here.

In [ ]:
comparison = json.loads(paths.COMPARISON_JSON.read_text())
print('split           :', comparison['split'])
print('device          :', comparison['device'])
print('git revision    :', comparison['git_revision'])
print('variants        :', list(comparison['variants']))


### 2.1 GED against sample count

The paper's qualitative claim is that GED falls as more samples are drawn. Sample
counts are prefixes of one 16-sample draw, so the trend is a within-sample comparison.

In [ ]:
counts = comparison['sample_counts']
fig, ax = plt.subplots(figsize=(6, 4))
for name, report in comparison['variants'].items():
    block = report['aggregate_over_all_patches']
    ax.plot(counts, [block[f'ged@{c}']['mean'] for c in counts], marker='o', label=name)
ax.set_xlabel('samples'); ax.set_ylabel(r'$D^2_{GED}$ (lower is better)')
ax.set_xscale('log', base=2); ax.set_xticks(counts); ax.set_xticklabels(counts)
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()


### 2.2 Single-sample quality against the degenerate baselines

The bar that matters for the extension: **random** is the floor, **oracle** the ceiling,
and **all-empty** is the trap. An all-empty predictor scores Dice 0.75 on the 33% of
patches where three of four graders are empty, so any single-sample number has to be
read against it rather than against zero.

In [ ]:
# TODO(phase 3): add the head-selected bar once selected_dice@n is present.
largest = max(counts)
labels = ['random', 'emptiest', 'oracle', 'all-empty']
keys = [f'random_sample_dice@{largest}', f'emptiest_sample_dice@{largest}',
        f'oracle_dice@{largest}', 'empty_dice']
fig, ax = plt.subplots(figsize=(7, 4))
width = 0.8 / max(len(comparison['variants']), 1)
for offset, (name, report) in enumerate(comparison['variants'].items()):
    block = report['aggregate_over_all_patches']
    ax.bar(np.arange(len(keys)) + offset * width,
           [block[k]['mean'] for k in keys], width, label=name)
ax.set_xticks(np.arange(len(keys)) + width / 2); ax.set_xticklabels(labels)
ax.set_ylabel('Dice'); ax.legend(); ax.grid(alpha=0.3, axis='y'); fig.tight_layout()


### 2.3 Per ambiguity bucket

Aggregate numbers are dominated by the lesion-presence question. Bucket k holds patches
where k of the four graders saw a lesion; bucket 1 is 33% of the data and the hard case
for the extension. Median lesion area rises with the bucket, so a poor bucket number
should be read against object size before concluding the model is worse there.

In [ ]:
rows = []
for name, report in comparison['variants'].items():
    for bucket, block in sorted(report['per_bucket'].items()):
        rows.append((name, bucket, block['n_patches'], block['lesion_area_median_px'],
                     block[f'ged@{largest}']['mean'], block[f'oracle_dice@{largest}']['mean'],
                     block['empty_dice']['mean']))
header = f"{'variant':<14}{'bucket':>7}{'patches':>9}{'medArea':>9}{'GED':>9}{'oracle':>9}{'empty':>8}"
print(header); print('-' * len(header))
for r in rows:
    print(f'{r[0]:<14}{r[1]:>7}{r[2]:>9}{r[3]:>9.1f}{r[4]:>9.4f}{r[5]:>9.4f}{r[6]:>8.4f}')


## 3. Qualitative panels

Each row is one validation patch: the image, its four grader masks, then samples drawn
from the prior. The patches are the stratified diagnostic set -- one per ambiguity
bucket -- and they are the same patches in every run, so panels are comparable across
variants and phases.

This reads the **tracked subset** export. `panel_batch` resolves the recorded global
indices through `source_index`, so the identical call works against the full dataset;
only `npz_path` changes.

In [ ]:
arrays = LidcArrays.load(paths.SUBSET_NPZ)
print('subset:', len(arrays), 'patches |', arrays.spatial_shape, '| is_subset:', arrays.is_subset)
panel_indices = arrays.source_index[:4]
images, graders = panel_batch(arrays, panel_indices)
print('images', tuple(images.shape), '| graders', tuple(graders.shape))


In [ ]:
# TODO: point at a downloaded weights-only export, e.g.
#   from probunet.evaluation.runner import load_variant
#   variant, config, state = load_variant(Path('baseline_best_weights.pt'), device)
#   samples = variant.sample(images.to(device), n_samples=6).cpu()
# Until then, show the graders alone so the layout is visible without any weights.
empty = torch.zeros(images.shape[0], 1, *arrays.spatial_shape, dtype=torch.uint8)
panel = make_panel(images, graders, empty)
fig, ax = plt.subplots(figsize=(10, 2.2 * images.shape[0]))
ax.imshow(panel[0], cmap='gray', vmin=0, vmax=1); ax.axis('off')
ax.set_title('image | 4 grader masks | prior samples'); fig.tight_layout()


## 4. Deviations and limitations

`DEVIATIONS.md` records every departure from the paper and the authors' released code,
with the expected impact of each. `data/splits/SPLIT_NOTES.md` records two accepted
properties of the split. Both are tracked in the repo and are the honest counterpart to
the numbers above.

In [ ]:
print(Path('DEVIATIONS.md').read_text()[:1500], '...')
